In [ ]:
import os
import json
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

# ==========================================
# 1. 초기 설정 및 클라이언트 초기화
# ==========================================

os.environ["OPENAI_API_KEY"] = "sk-" # 여기에 본인 API 키 입력
client = OpenAI()

# 데이터 경로 (JSON 파일들이 있는 폴더 경로)
DATA_PATH = r"C:\Users\sunhe\OneDrive\문서\GitHub\Card-Recommendation-Chatbot\data\check_cards_benefits"
ko_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

# Vector DB 설정
chroma_client = chromadb.PersistentClient(path="./card_vector_db")

# 신규 컬렉션 생성을 위해 기존 것 삭제 (테스트용)
try:
    chroma_client.delete_collection(name="card_chunked_collection")
except:
    pass

collection = chroma_client.create_collection(
    name="card_chunked_collection",
    embedding_function=ko_embedding_func
)

# ==========================================
# 2. 청킹 관련 함수 정의 (추가된 부분)
# ==========================================
def split_by_newline(text):
    """줄바꿈 기준으로 리스트화"""
    return [line.strip() for line in text.split("\n") if line.strip()]

def split_by_space_with_limit(text, max_len=200):
    """글자수 제한에 맞춰 공백 기준으로 분할"""
    words = text.split(" ")
    chunks = []
    current = ""
    for word in words:
        if len(current) + len(word) + 1 <= max_len:
            current += (" " if current else "") + word
        else:
            chunks.append(current)
            current = word
    if current: chunks.append(current)
    return chunks

# ==========================================
# 3. 데이터 로드 및 청킹 인덱싱 함수 (추가된 부분)
# ==========================================
def indexing_with_chunks():
    if not os.path.exists(DATA_PATH):
        print(f"❌ 경로를 찾을 수 없습니다: {DATA_PATH}")
        return

    file_list = [f for f in os.listdir(DATA_PATH) if f.endswith('.json')]
    all_chunks_text = []
    all_metadatas = []
    all_ids = []
    
    id_counter = 0
    # 청킹 설정값
    MAX_CHUNK_SIZE = 300  # 청크 당 최대 글자 수 (약간 늘림)
    OVERLAP_SIZE = 50     # 문맥 유지를 위해 겹칠 글자 수

    print(f"📂 [{DATA_PATH}]에서 고도화된 청킹 및 인덱싱 시작...")

    for file_name in file_list:
        with open(os.path.join(DATA_PATH, file_name), 'r', encoding='utf-8') as f:
            card = json.load(f)
            
            card_name = card.get("card_name", "")
            company = card.get("company", "")
            performance = card.get("performance", "")
            overseas = card.get("overseas", "")
            
            # 카드의 전체적인 특징을 요약한 헤더 (모든 청크에 삽입)
            card_header = f"카드명: {card_name} | 카드사: {company} | 전월실적: {performance}"

            for benefit in card.get("benefit", []):
                category = benefit.get("category", "")
                content = benefit.get("content", "").strip()

                # 1단계: 의미 단위(카테고리별)로 먼저 접근
                # 2단계: 내용이 길 경우 슬라이딩 윈도우 적용
                start = 0
                while True:
                    # 중첩을 고려하여 서브 텍스트 추출
                    end = start + MAX_CHUNK_SIZE
                    sub = content[start:end]
                    
                    if not sub:
                        break

                    # 조각 데이터 구성 (컨텍스트 강화)
                    chunk_text = (
                        f"{card_header}\n"
                        f"혜택분류: {category}\n"
                        f"상세내용: {sub}\n"
                        f"해외사용여부: {overseas}"
                    )
                    
                    all_chunks_text.append(chunk_text)
                    all_metadatas.append({
                        "name": card_name, 
                        "company": company,
                        "category": category
                    })
                    all_ids.append(f"chunk_{id_counter}")
                    id_counter += 1

                    # 끝에 도달했으면 종료
                    if end >= len(content):
                        break
                    
                    # [핵심] OVERLAP 적용: 다음 시작 지점을 중첩만큼 뒤로 당김
                    start += (MAX_CHUNK_SIZE - OVERLAP_SIZE)

    # 벡터 DB 저장 (Batch 처리로 안정성 확보)
    batch_size = 100
    for i in range(0, len(all_chunks_text), batch_size):
        collection.add(
            ids=all_ids[i:i+batch_size],
            documents=all_chunks_text[i:i+batch_size],
            metadatas=all_metadatas[i:i+batch_size]
        )
        
    print(f"✅ 총 {len(all_chunks_text)}개의 고도화된 데이터 조각이 DB에 등록되었습니다.")

# ==========================================
# 4. 프롬프트 엔지니어링 (TOP 3 추천 로직)
# ==========================================
def get_model_response(query, persona_dict, use_rag=True):
    persona_name = persona_dict['name']
    persona_traits = persona_dict['traits']

    if use_rag:
        # 1. 검색 쿼리 강화 및 결과 추출 (n_results 상향)
        search_query = f"{persona_traits} {query}"
        results = collection.query(query_texts=[search_query], n_results=40)
        raw_documents = results['documents'][0]
        
        # 2. 검색된 조각들로부터 '실제 존재하는 카드 리스트' 추출 (중복 제거)
        available_card_names = set()
        card_context_map = {}
        
        for doc in raw_documents:
            # "카드명: XXX | 카드사: YYY" 형식에서 이름 추출
            first_line = doc.split("\n")[0]
            card_title = first_line.replace("카드명: ", "").split("|")[0].strip()
            
            available_card_names.add(card_title)
            
            if card_title not in card_context_map:
                card_context_map[card_title] = []
            card_context_map[card_title].append(doc)

        # 리스트를 문자열로 변환
        verified_list_str = ", ".join(list(available_card_names))

        formatted_context = ""
        for title, contents in card_context_map.items():
            formatted_context += f"\n### 카드 데이터: {title} ###\n"
            formatted_context += "\n".join(contents) + "\n"

        # 3. [초강력 제약 조건]이 포함된 프롬프트
        system_prompt = f"""
        당신은 금융 데이터 검증 전문가입니다. 
        반드시 아래의 [검증된 카드 리스트]에 이름이 있는 카드만 추천해야 합니다. 

        [검증된 카드 리스트 (이 리스트 외 추천 절대 금지)]
        {verified_list_str}

        [지식 베이스 상세 데이터]
        {formatted_context}

        [사용자 페르소나]
        - 이름: {persona_name}
        - 특성: {persona_traits}

        ────────────────
        ⛔ 절대 규칙 (위반 시 답변 무효)
        ────────────────
        1. [검증된 카드 리스트]에 없는 카드 이름은 당신의 머릿속에 있더라도 절대로 언급하지 마십시오.
        2. 리스트에 있는 카드 중 페르소나와 가장 잘 맞는 카드를 최대 3개 선정하십시오.
        3. 카드명과 카드사 이름을 지식 베이스에 있는 그대로(토씨 하나 틀리지 않게) 출력하십시오.
        4. 만약 리스트에 적합한 카드가 없다면 지어내지 말고, 리스트 내에서 가장 유사한 혜택을 가진 카드를 차선책으로 추천하십시오.

        [출력 형식]
        ### 🏆 데이터 기반 맞춤 카드 추천 TOP 3
        
        1위: [정확한 카드명] : [정확한 카드사]
        - 혜택 근거: (데이터 기반 수치 포함)
        - 선정 이유: (페르소나 맞춤 분석)
        ...
        """
    else:
        system_prompt = f"상담사로서 {persona_name}({persona_traits})에게 실존하는 카드 3개를 추천하세요."

    # GPT-4o-mini 모델 사용 (지시 준수율이 3.5보다 압도적으로 높음)
    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content
# ==========================================
# 5. 실행 및 결과 출력
# ==========================================
def run_evaluation_display(scenarios):
    print("\n" + "🚀" * 30)
    print("   카드 추천 모델 성능 비교 테스트 시작")
    print("🚀" * 30 + "\n")

    for i, sc in enumerate(scenarios):
        print(f"📍 [테스트 시나리오 {i+1}]")
        print(f"👤 페르소나: {sc['persona']['name']} ({sc['persona']['traits']})")
        print(f"❓ 질문: {sc['query']}")
        print("-" * 60)

        print("\n[1. GPT-3.5 답변 (기본 지식)]")
        print(get_model_response(sc['query'], sc['persona'], use_rag=False))

        print("\n" + "." * 40)

        print("\n[2. RAG 모델 답변 (데이터 기반)]")
        print(get_model_response(sc['query'], sc['persona'], use_rag=True))

        print("\n" + "=" * 80 + "\n")

# --- 실행 순서 ---
indexing_with_chunks()  # 1. 데이터를 쪼개서 DB에 넣기

test_scenarios = [
    {
        "persona": {
            "name": "20대 자기계발생",
            "traits": "무실적 카드 선호, 대중교통 및 학원 이용 잦음, 편의점/카페 지출 위주, 온라인 쇼핑 및 다이소 등 생활지출"
        },
        "query": "전월 실적 부담 없으면서 대중교통, 편의점, 학원 할인이 잘 되는 카드를 순위별로 추천해줘."
    },
    {
        "persona": {
            "name": "30대 직장인",
            "traits": "안정적인 급여, 자차 주유 혜택 필요, 해외 직구 및 여행 결제 적립, 병원 및 헬스장 등 건강관리 지출"
        },
        "query": "주유 혜택과 해외 적립이 강력하고 건강관리 지출에도 도움되는 혜택 많은 카드를 추천해줘."
    },
    {
        "persona": {
            "name": "40대 가장",
            "traits": "자녀 학원비 결제 비중 높음, 대형마트 장보기, 공과금 및 관리비 자동이체 필수"
        },
        "query": "학원비 할인율이 가장 높고 마트와 공과금 혜택이 포함된 생활밀착형 카드를 알려줘."
    }
]

run_evaluation_display(test_scenarios)  # 2. 결과 출력